# Antispoofing

In this homework, you will develop a countermeasure against deepfakes and then try to explain it using various XAI techniques.

More specifically, you will implement and train a Countermeasure (CM) system on the Logical Access partition of the [ASVSpoof 2019 Dataset](https://datashare.ed.ac.uk/handle/10283/3336) ([Kaggle Link](https://www.kaggle.com/datasets/awsaf49/asvpoof-2019-dataset)). You may find the [ASVspoof 2019 evaluation plan](https://www.asvspoof.org/asvspoof2019/asvspoof2019_evaluation_plan.pdf) useful.

For the CM, we choose [LightCNN (LCCN)](https://arxiv.org/abs/1511.02683) that once achieved the top place in the competition. We will follow the Speech Technology Center (STC) [paper](https://arxiv.org/abs/1904.05576).

**Hints**:

1. Use STFT (FFT in the paper) as front-end.

2. The dropout layer is put before the last batch norm.

In [4]:
import os
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset

import torchaudio
import torchaudio.transforms as T

## Dataset [0.5 pts]

We want to train a neural network to predict if the input audio is real or fake. To do so, we need a dataset first. In this homework, we will work with [ASVspoof19](https://arxiv.org/pdf/1911.01601.pdf).

Create a `Dataset` class that downloads the dataset, parses its metadata and, given index $i$, returns $i$-th object of the dataset. Do not forget to preprocess audio for LCNN (calculate stft, etc.).

In [ ]:
# !pip install kagglehub

In [1]:
import kagglehub

# Download latest version
kagglehub.dataset_download("awsaf49/asvpoof-2019-dataset")

Resuming download from 14405337088 bytes (10916486056 bytes left)...
Resuming download to C:\Users\goryachev\.cache\kagglehub\datasets\awsaf49\asvpoof-2019-dataset\1.archive (14405337088/25321823144) bytes left.


100%|██████████| 23.6G/23.6G [33:59<00:00, 5.35MB/s]  

Extracting files...


'C:\\Users\\goryachev\\.cache\\kagglehub\\datasets\\awsaf49\\asvpoof-2019-dataset\\versions\\1'

In [6]:
class ASVSpoof2019(Dataset):
    """
    ASVspoof2019 Logical Access Dataset
    optimized for Kaggle usage.

    Returns:
        {
            "features": Tensor [1, freq, time],
            "label": Tensor,
            "utt_id": str
        }
    """

    PARTITIONS = {
        "train": {
            "folder": "ASVspoof2019_LA_train",
            "protocol": "ASVspoof2019.LA.cm.train.trn.txt",
        },
        "dev": {
            "folder": "ASVspoof2019_LA_dev",
            "protocol": "ASVspoof2019.LA.cm.dev.trl.txt",
        },
        "eval": {
            "folder": "ASVspoof2019_LA_eval",
            "protocol": "ASVspoof2019.LA.cm.eval.trl.txt",
        },
    }

    def __init__(
        self,
        root_dir="./data",
        split="train",
        sample_rate=16000,
        n_fft=1728,
        hop_length=130,
        win_length=1728,
        max_frames=750,
    ):
        super().__init__()

        if split not in self.PARTITIONS:
            raise ValueError(
                f"split must be one of {list(self.PARTITIONS.keys())}"
            )

        self.root_dir = Path(root_dir)
        self.split = split
        self.sample_rate = sample_rate
        self.max_frames = max_frames

        # -----------------------------
        # Dataset paths
        # -----------------------------
        split_info = self.PARTITIONS[split]

        self.base_dir = self.root_dir / split_info["folder"]

        self.audio_dir = self.base_dir / "flac"

        protocol_path = (
            self.base_dir
            / "ASVspoof2019_LA_cm_protocols"
            / split_info["protocol"]
        )

        # -----------------------------
        # STFT frontend
        # -----------------------------
        self.spec_transform = T.Spectrogram(
            n_fft=n_fft,
            win_length=win_length,
            hop_length=hop_length,
            power=2,
        )

        self.resampler = None

        # -----------------------------
        # Metadata
        # -----------------------------
        self.metadata = self._load_protocol(protocol_path)

    def _load_protocol(self, protocol_path):
        """
        Protocol format:
        speaker_id utt_id - attack_id label
        """

        records = []

        with open(protocol_path, "r") as f:
            for line in f:
                parts = line.strip().split()

                records.append(
                    {
                        "speaker_id": parts[0],
                        "utt_id": parts[1],
                        "attack_id": parts[3],
                        "label": parts[4],
                    }
                )

        return pd.DataFrame(records)

    def __len__(self):
        return len(self.metadata)

    def _load_waveform(self, utt_id):
        audio_path = self.audio_dir / f"{utt_id}.flac"

        waveform, sr = torchaudio.load(audio_path)

        # Convert stereo -> mono
        if waveform.size(0) > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Resample if needed
        if sr != self.sample_rate:

            if self.resampler is None:
                self.resampler = T.Resample(sr, self.sample_rate)

            waveform = self.resampler(waveform)

        return waveform

    def _extract_features(self, waveform):
        """
        Log-power spectrogram for LCNN.
        """

        spec = self.spec_transform(waveform)

        # Log compression
        spec = torch.log(spec + 1e-6)

        # Utterance normalization
        spec = (spec - spec.mean()) / (spec.std() + 1e-6)

        # Fixed-length padding/cropping
        time_dim = spec.size(-1)

        if time_dim < self.max_frames:

            pad_size = self.max_frames - time_dim

            spec = F.pad(
                spec,
                (0, pad_size),
                mode="constant",
                value=0,
            )

        else:
            spec = spec[:, :, :self.max_frames]

        return spec

    def __getitem__(self, idx):

        item = self.metadata.iloc[idx]

        utt_id = item["utt_id"]

        waveform = self._load_waveform(utt_id)

        features = self._extract_features(waveform)

        label = 0 if item["label"] == "bonafide" else 1

        return {
            "features": features,
            "label": torch.tensor(label, dtype=torch.long),
            "utt_id": utt_id,
        }

**Hint**: when working in Kaggle, it is easier and faster to use dataset as kaggle input. We can use it directly or add a symlink to a local dir using `ln -s`.
**Hint**: it might be easier to do this homework in Kaggle, since model training may take some time

Create train/eval dataset and dataloaders:

In [8]:
from torch.utils.data import DataLoader

# -----------------------------
# Datasets
# -----------------------------
train_dataset = ASVSpoof2019(
    root_dir="./data",
    split="train",
)

eval_dataset = ASVSpoof2019(
    root_dir="./data",
    split="eval",
)

print(f"Train size: {len(train_dataset)}")
print(f"Eval size:  {len(eval_dataset)}")


# -----------------------------
# Dataloaders
# -----------------------------
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

eval_loader = DataLoader(
    eval_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)


# -----------------------------
# Sanity check
# -----------------------------
batch = next(iter(train_loader))

print("Features:", batch["features"].shape)
print("Labels:  ", batch["label"].shape)
print("IDs:     ", batch["utt_id"][:3])

FileNotFoundError: [Errno 2] No such file or directory: 'data\\ASVspoof2019_LA_train\\ASVspoof2019_LA_cm_protocols\\ASVspoof2019.LA.cm.train.trn.txt'

Visualize one object, just to check that all is fine:

In [ ]:
# YOUR CODE HERE

## Loss function [0.5 pts]

In the lecture, we saw different softmax losses and the motivation behind them for the ASV task. However, they can also be used for any classification task, such as synthesized speech detection. The papers suggest to use A(M)-Softmax or Cross-Entropy. The STC paper argues that A-Softmax is better.

(a) Explain what are the benefits of A-softmax over cross-entropy according to the STC paper?

(b) Analyse the [NII paper](https://arxiv.org/pdf/2103.11326) and explain if complicated Softmax is actually needed to achieve good EER or we can go with Cross Entropy.

## **Answers**: 
### (a) Benefits of A-Softmax over Cross-Entropy according to the STC paper

The STC Antispoofing Systems for the ASVspoof2019 Challenge argues that Angular Softmax (A-Softmax, also called angular-margin Softmax) improves the discriminative power of embeddings compared to standard cross-entropy (CE).

The main idea is that:
* Cross-Entropy only separates classes linearly
* A-Softmax additionally enforces angular margins between classes

For anti-spoofing, this matters because:
* bona fide and spoofed speech can be very similar acoustically
* unseen attacks during evaluation differ from training attacks
* the model must generalize beyond known spoofing methods

The STC paper claims A-Softmax helps produce:
* more compact intra-class embeddings
* larger inter-class separation
* better generalization to unseen attacks

Mathematically: standard CE uses logits 
$$
W^{T} \cdot x + b
$$. 

A-Softmax normalizes the features and classifier weights and introduces an angular margin
$$
cos(mθ),
$$
where:
* θ = angle between feature vector and class weight
* m = angular margin

This creates stricter decision boundaries. The practical consequences described in the paper are:
| Cross Entropy                           | A-Softmax                              |
| --------------------------------------- | -------------------------------------- |
| Learns separable classes                | Learns angularly separable classes     |
| Decision boundary is weaker             | Margin-based boundary is stricter      |
| More prone to overfitting known attacks | Better generalization                  |
| Embeddings less discriminative          | Embeddings more compact/discriminative |


The STC system achieved very low EERs using LCNN + A-Softmax:
* 1.86% EER for Logical Access
* 0.54% EER for Physical Access

The paper attributes part of this improvement to the angular-margin loss.

### (b) Does the NII paper show that complicated Softmax is actually necessary?

The A Comparative Study on Recent Neural Spoofing Countermeasures for Synthetic Speech Detection provides a much broader and more careful comparison of:
* different front-ends
* architectures
* pooling methods
* loss functions

on [ASVspoof2019 LA](https://cir.nii.ac.jp/crid/1360013168873609344).

Their conclusion is much more nuanced than the STC paper. 

The key takeaway is complicated margin-based Softmax losses are not always necessary to achieve strong anti-spoofing performance.

The NII study found that:
1. improvements from margin-based losses were often small
2. results depended heavily on:
3. frontend features
4. architecture
5. score aggregation
6. random initialization
7. standard CE frequently performed competitively

In many experiments:
* CE achieved nearly the same EER as A-Softmax variants
* variance between runs could be comparable to the gain from fancy losses

The paper emphasizes that:
* frontend representation matters more than loss complexity
* model stability/reproducibility matters
* some published gains from special losses may be overstated

Their analysis suggests that:
* a good architecture + good preprocessing + CE can already achieve excellent EER
* margin-based Softmax is helpful but not essential

This is especially true when using stronger modern models.
The practical interpretation is:


Following tha NII paper, we will continue with Cross Entropy

In [ ]:
from torch import nn

criterion = nn.CrossEntropyLoss()

## Evaluation metric [0.5 pts]

We will use equal error rate as the primary evaluation metric. The code for calculating metrics is provided by the ASVspoof itself. We just need to write a wrapper. Given model logits and labels, calculate EER using the ASVspoof functions.

Your model returns two probas: [spoof_proba, bona_proba]. Be careful with the EER metric and recal how ROC curve is computed to ensure that you do not make a mistake.

In [ ]:
# download asvspoof metric calculation functions
!wget https://raw.githubusercontent.com/markovka17/dla/refs/heads/2023/hw5_as/calculate_eer.py

In [ ]:
from calculate_eer import compute_eer

def get_eer(logits, labels):
    # YOUR CODE HERE

## LCNN Implementation [6.0 pts]

Create a `LCNN` class for the model architecture.

In [ ]:
# YOUR CODE HERE

In [ ]:
model = LCNN(...)

In [ ]:
# double-check that it runs (do eval mode)
# YOUR CODE HERE

Write the train loop. Since it may take some time, we advise you to save your checkpoints after each epoch to load it back if needed.

Plot EER vs epoch and loss vs epoch curves

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, scheduler, device):
    # YOUR CODE HERE


def evaluate(model, dataloader, criterion, device):
    # YOUR CODE HERE


def train(model, train_dataloader, eval_dataloader, criterion, optimizer, scheduler, device, n_epochs):
    # YOUR CODE HERE


In [ ]:
device = # YOUR CODE HERE

In [ ]:
# take optimizer and scheduler from the NII paper
optimizer = # YOUR CODE HERE
scheduler = # YOUR CODE HERE
n_epochs = # YOUR CODE HERE
train(model, train_dataloader, eval_dataloader,
      criterion, optimizer, scheduler, device, n_epochs)

The task is consired solved if you achieve at least $9\%$ EER. It is much higher than the model can achieve but we do not want you to wait 12+ hours for the model to converge.

## XAI [See points below]

Let's analyse the model we have created. We won't be able to understand the differences easily without having some reference. So, we will use the novel idea from the recent [Interspeech 2025 paper](https://arxiv.org/abs/2506.03425).

We will use a [vocoded dataset](https://arxiv.org/abs/2210.10570) of parallel samples: real and fake audio have the same speaker saying the same content at the same time. The ground-truth explanation will be obtained by calculating difference between real and fake spectrograms.

In [ ]:
!wget https://zenodo.org/records/7314976/files/project09-voc.v4.tar?download=1 -O project09-voc.v4.tar
!tar -xvf project09-voc.v4.tar

Note that real audio is taken from ASVspoof. So let's take a real example from the asvspoof dataset. Using its filename, find the corresponding `hifi-gan` and `waveglow` vocoded versions in the vocv4 and load them too

In reality, we are interested in the explanations for the unseen data. But for this homework, let's consider the train set. This will allow us to see if the model learns the futures we expect it to learn (assuming the XAI tool is trustworthy) (Though spoof part of vocv4 is not exactly the same as the one in asvspoof, so we mostly eliminate the issues related to changing speakers, not algorithms)

In [ ]:
ind = # for consistency with solutions choose the index that corresponds to LA_T_4179989 (bona fide)
# YOUR CODE HERE

In [ ]:
# get LCNN-prepared spectrogram for the paired real and fake example from the dataset
# paired: the same filename, but one is bona fide another is created via vocoder
real_audio = # YOUR CODE HERE
hifigan_audio = # YOUR CODE HERE
waveglow_audio = # YOUR CODE HERE


# fake audio may be slightly longer due to padding, remove some part from the end to make the length equal
# YOUR CODE HERE


# preprocess audio for model input
# YOUR CODE HERE

Run your model on these clips. See if the model prediction is correct. Use this understanding for the following analysis

In [ ]:
# YOUR CODE HERE

### Manual explanation [0.5 pts]

Compare the two spectrograms (real and fake). What differences do you see? (**Hint**: they exist, if you do not see -- look carefully).

In [ ]:
# YOUR CODE HERE

**Your answer here**

### Automatic explanation. [0.5 pts]

Calculate Eq. 2 from the [Interspeech 2025 paper](https://arxiv.org/abs/2506.03425) to automatically highlight the differences between two objects

In [ ]:
# YOUR CODE HERE

Plot the mask on top of the fake spectrogram and compare three plots: real, fake, fake+mask on top. Do it for both vocoders. Compare

In [ ]:
# YOUR CODE HERE

**Your comparison of the ground-truth mask with your manual analysis (from previous subtask) here**

### Grad-CAM [0.5 pts]

Using [pytorch-grad-cam lib](https://github.com/jacobgil/pytorch-grad-cam), implement [Grad-CAM](https://arxiv.org/abs/1610.02391) for your LCNN model. Choose the layer you like

In [ ]:
# grad-cam and captum may have conflicting numpy dependencies. Just install grad-cam first, then captum and it will work

In [ ]:
!pip install grad-cam

In [ ]:
# YOUR CODE HERE

### Comparison [1.0 pts]

Compare your Grad-CAM attributions with another gradient-based method: InputXGradient. Compute it using [Captum](https://captum.ai/).

In [ ]:
!pip install captum

In [ ]:
# YOUR CODE HERE

Do three plots: mask vs grad-cam vs inputXgradient. Compare them. Does any of the XAI methods align with the mask?

In [ ]:
# YOUR CODE HERE

**Your answer here**

Due to skewed distribution for some XAI tools, you may want to look at top-5% points, similarly to the ground-truth mask. Binarize attributions using their $95\%$ quantile and plot again:

In [ ]:
# YOUR CODE HERE

**Your analysis here**

http://github.com/LauzHack/deep-learning-bootcamp/blob/summer25/day08/Seminar_AntiSpoofing.ipynb